In [1]:
import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings('ignore')

In [40]:
yerevan_historical = pd.read_csv('../data/yerevan_historical_clean/sale_historical.csv')
yerevan_actual = pd.read_csv('../data/yerevan_actual_clean/sale_actual.csv')

## Modifying historical data columns

In [41]:
yerevan_historical.drop(columns=['sold_date', 'days_in'], inplace=True)

In [42]:
yerevan_actual['backup_status'] = 'not sold'

In [43]:
df = pd.concat([yerevan_historical, yerevan_actual], axis=0)

In [112]:
df[df['id']==10895]

,backup_status,id,price,rooms,square_meters,address,source,furniture,renovation,price_per_meter,...,links,download_date,y,x,closest_metro,walking_distance_to_metro(m),district,latitude_jittered,longitude_jittered,neighborhood
18223,not sold,10895,125000.0,3.0,54.0,"Yerevan, Center, Tigran Mets Ave",real-estate.am,1.0,NaN,2314.814815,...,https://www.real-estate.am/en/2-bedrooms/apart...,2023-03-13 18:12:05,44.513112,40.166182,Zoravar Andranik,669,Kentron,44.51311,40.166191,Kentron


In [45]:
df = df.drop_duplicates(subset=['id'], keep='last')
df = df.loc[~((df['backup_status'] == 'sold') & (df['id'].duplicated()))]

In [46]:
df = df.dropna(subset=['district']).reset_index(drop=True)

## Price change table

In [9]:
df1 = pd.concat([yerevan_historical, yerevan_actual], axis=0)

In [10]:
df1[df1['id']==10895]

,backup_status,id,price,rooms,square_meters,address,source,furniture,renovation,price_per_meter,...,links,download_date,y,x,closest_metro,walking_distance_to_metro(m),district,latitude_jittered,longitude_jittered,neighborhood
33,sold,10895,100000.0,3.0,54.0,"Yerevan, Center, Tigran Mets Ave",real-estate.am,1.0,NaN,1851.851852,...,https://www.real-estate.am/en/2-bedrooms/apart...,2022-08-28 19:49:22,44.513112,40.166182,Zoravar Andranik,669,Kentron,44.513121,40.166179,NaN
18235,not sold,10895,125000.0,3.0,54.0,"Yerevan, Center, Tigran Mets Ave",real-estate.am,1.0,NaN,2314.814815,...,https://www.real-estate.am/en/2-bedrooms/apart...,2023-03-13 18:12:05,44.513112,40.166182,Zoravar Andranik,669,Kentron,44.513110,40.166191,Kentron


In [11]:
df1.loc[df1.duplicated(subset=['id'], keep=False), 'backup_status'] = 'price change'
df1 = df1.drop_duplicates(subset=['id'], keep='last')

In [12]:
df1 = df1.dropna(subset=['district']).reset_index(drop=True)

### classifying neighborhoods

In [55]:
df2 = df.head(80589).reset_index(drop=True)

In [32]:
import requests

api_key = "***REMOVED-GOOGLE-API-KEY***"

addresses = []

In [33]:
while True:
    try:
        for index, row in df2.iterrows():
            latlng = f"{row['x']},{row['y']}"
            url = f"https://maps.googleapis.com/maps/api/geocode/json?latlng={latlng}&location_type=APPROXIMATE&key={api_key}"
            response = requests.get(url)
            data = response.json()
            address = data['results'][0]['address_components'][0]['long_name']
            # print(address)
            df2.drop(index=df2.index[0], axis=0, inplace=True)
            addresses.append(address)
    except:
        # if there was an error, print a message and try again
        print("An error occurred. Retrying...")
        continue
    else:
        # if there was no error, break out of the loop and continue with the rest of the code
        break

An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error

In [87]:
df['neighborhood'] = addresses

### small amount of rows that have yerevan classfied replaced with district value 

In [89]:
for index, row in df[df['neighborhood'] == 'Yerevan'].iterrows():
    df.loc[index, 'neighborhood'] = row['district']

In [90]:
df.loc[df['neighborhood'] == '0054', 'neighborhood'] = 'Davtashen Block'
df.loc[df['neighborhood'] == '0059', 'neighborhood'] = 'Nor Nork 9th Microdistrict'

df['neighborhood'] = df['neighborhood'].str.replace(' ', '_')
df['neighborhood'] = df['neighborhood'].str.replace('-', '_')

In [93]:
sale_historical = df[df['backup_status']=='sold'].reset_index(drop=True)
sale_actual = df[df['backup_status']=='not sold'].reset_index(drop=True)

In [94]:
# removing neighborhoods with low counts
neighborhood_counts = sale_historical['neighborhood'].value_counts()
neighborhood_df = pd.DataFrame({'neighborhood': neighborhood_counts.index, 'count': neighborhood_counts.values})
neighborhoods_to_remove = neighborhood_df[neighborhood_df['count'] < 5]['neighborhood'].tolist()
sale_historical = sale_historical[~sale_historical['neighborhood'].isin(neighborhoods_to_remove)]

In [95]:
# removing neighborhoods with low counts
neighborhood_counts = sale_actual['neighborhood'].value_counts()
neighborhood_df = pd.DataFrame({'neighborhood': neighborhood_counts.index, 'count': neighborhood_counts.values})
neighborhoods_to_remove = neighborhood_df[neighborhood_df['count'] < 5]['neighborhood'].tolist()
sale_actual = sale_actual[~sale_actual['neighborhood'].isin(neighborhoods_to_remove)]

In [96]:
sale_historical.to_csv(r'../data/yerevan_combined_clean/historical_sold.csv', index=False)
sale_actual.to_csv(r'../data/yerevan_combined_clean/actual_not_sold.csv', index=False)

In [106]:
df = pd.concat([sale_historical, sale_actual], axis=0)

In [107]:
df_clean = df.dropna().reset_index(drop=True)

In [111]:
df.to_csv(r'../data/yerevan_combined_clean/sale_combined.csv', index=False)
df_clean.to_csv(r'../data/yerevan_combined_clean/sale_clean_combined.csv', index=False)
df1.to_csv(r'../data/yerevan_combined_clean/sale_combined_price_change.csv', index=False)